# Lab 00-00: AWS Deployment & Cost Guardrails**Module:** 00 — Foundations  **Exam weight:** Foundational (not directly tested, but you need a working workspace for every lab)  **UI Sections:** AWS Console | Databricks Account Console | Workspace Settings  **Estimated Time:** 30–45 minutes  **Difficulty:** Beginner---## OverviewBefore you can write a single line of code, you need a **running Databricks workspace**.This lab walks you through deploying Databricks on AWS via the Marketplace free trialand — critically — setting up **cost guardrails** so you don't burn through your $400in credits on day one.> **Why this lab exists:** The exam assumes you have hands-on experience with the full> Databricks platform. The AWS Marketplace trial gives you the full platform for 14 days> with $400 in credits. That's plenty for all 35 labs — **if** you manage costs wisely.---## What You Will Set Up| Component | Purpose | Estimated Cost ||-----------|---------|---------------|| Databricks workspace | Your development environment | Included in trial || Single-node cluster | Run Python/Spark notebooks | ~$0.50–1.00/hour || Serverless compute | Quick notebook runs | ~$0.07/DBU || Vector Search endpoint | RAG retrieval (1 unit) | ~$0.38/hour when active || Model Serving | Foundation Model API calls | Pay-per-token (cheap) |**Total estimated cost for all 35 labs:** $80–150 (well within $400 trial)---

---## Step 1: Subscribe via AWS Marketplace### Prerequisites- An **AWS account** with a valid payment method (credit card)- You will NOT be charged during the trial unless you exceed $400- After 14 days, it converts to pay-as-you-go — you can cancel anytime---### Phase 1: Subscribe1. Go to the [Databricks AWS Marketplace listing](https://aws.amazon.com/marketplace/pp/prodview-wtyi47bkbkb7a)2. Click **"Try for free"**3. You'll land on the **Subscribe** page. Review:   - **Pricing:** $1.00 per Databricks Consumption Unit (DBU)   - **Free trial:** Up to $400 in credits during 14-day trial   - **After trial:** Automatic enrollment in pay-as-you-go (cancel anytime)4. Leave **"No purchase order"** selected (you don't need one)5. Click **Subscribe**6. Accept the **EULA** (End User License Agreement)> **VAT Note:** If you're billed through AWS EMEA (e.g., Netherlands), there's a> 21% VAT on top. This doesn't reduce your $400 credit — the credit covers the> base DBU cost, and VAT is separate on your AWS invoice.---### Phase 2: Create and Link Your Databricks AccountAfter subscribing, you'll see a page saying **"Verify permissions and link Databricks account"**.1. Review the **IAM permissions** shown — these are what AWS needs to create   infrastructure (VPCs, S3 buckets, IAM roles) for your workspace. If you're the   AWS account owner, you already have these permissions.2. Click **"Create account"** — this opens a new tab on the Databricks website3. On the Databricks site, fill in:   - **Email address** (use your real email — this is your login)   - **Full name**   - **Password** (choose a strong one)   - **Company name** (put "Personal" or your name)4. Complete the Databricks account creation5. Go back to the **AWS tab** — it should now show **"Status: Linked"**6. You'll see a confirmation: **"Your AWS subscription was successfully linked"**7. Click **"Continue to your Databricks account"** or click **Next** on the AWS page---### Phase 3: Configure Your WorkspaceYou'll land on the **"Configure workspace"** page with three sections:#### Amazon S3 and IAM Resources| Setting | What to Do ||---------|-----------|| **AWS Region** | Pick the region closest to you. Default is `US East (N. Virginia)`. EU options: `EU (Ireland)`, `EU (Frankfurt)` || **S3 bucket name** | Leave the auto-generated name (e.g., `databricks-XXXX-cloud-storage-bucket`) || **IAM role for storage** | Leave the auto-generated name || **IAM role for compute** | Leave the auto-generated name || **Existing S3 bucket** | Leave as **"Not specified"** — you don't need one |> **Rule of thumb:** Don't change any auto-generated names unless you have a specific reason.#### VPC ResourcesLeave all VPC configurations at their defaults. Databricks will create a dedicated VPCwith private subnets, NAT gateway, and security groups automatically.#### Acknowledgement**Check the box** that says you acknowledge Databricks will create resources in yourAWS account. This is required — it confirms you understand that AWS infrastructure(VPC, S3, IAM) will be created, which may incur minor AWS charges (typically < $5/monthfor the VPC NAT gateway).Click **Next**.---### Phase 4: Review and LaunchYou'll see a summary of everything:- **Step 1:** Databricks account — Status: Linked- **Step 2:** Workspace config — region, S3 bucket, IAM rolesReview that everything looks correct, then click **"Launch product"**.**Wait 5–10 minutes** while AWS provisions:- A VPC with subnets and security groups- An S3 bucket for workspace storage- IAM roles for cross-account access- The Databricks workspace itselfOnce complete, you'll get a **workspace URL** like:`https://dbc-XXXXXXXX-XXXX.cloud.databricks.com`**Bookmark this URL** — you'll use it every day for the next 14 days.---### Phase 5: First Login1. Open your workspace URL2. Log in with the email and password you created in Phase 23. You should see the **Databricks Home page** with:   - A welcome banner   - Quick links to create notebooks, import data, etc.   - The left-hand navigation menu**You're in!** Your $400 trial is now active. The clock starts ticking from yourfirst compute usage (not from subscription time), so don't worry — just readingthe UI doesn't cost anything.> **Important:** Set a **calendar reminder for day 12** to review your usage and> decide whether to cancel or continue on pay-as-you-go.---

---## Step 2: Understand the Cost ArchitectureBefore configuring anything, you need to understand **what costs money** in Databricks.### The DBU (Databricks Unit)Everything in Databricks is billed in **DBUs** (Databricks Units). Think of a DBU like a"compute minute" — it measures how much processing power you used.> **Analogy: DBUs are like electricity units on your power bill.**> Running a small lamp (single-node cluster) uses fewer units per hour than running> an industrial oven (multi-node GPU cluster). The meter is always running when> the appliance is ON.### What Consumes DBUs| Resource | DBU Rate | Auto-Stops? | Your Control ||----------|----------|-------------|-------------|| **All-Purpose Cluster** | 0.4–4.0 DBU/hr (depends on size) | Only if auto-terminate is set | **HIGH** — terminate when done! || **Serverless Notebook** | 0.07 DBU/min (only while running) | Yes — stops when idle | Medium — auto-managed || **SQL Warehouse** | 0.22–0.38 DBU/min | Yes — auto-stops after idle | Medium || **Vector Search** | 0.38 DBU/hr per unit | **NO** — runs 24/7 until deleted | **HIGH** — delete when not using! || **Model Serving** | Per-token (Foundation Models) or DBU/hr (custom) | Scales to zero for Foundation Models | Low — pay per use || **Jobs/Workflows** | 0.07–0.25 DBU/hr | Yes — runs and stops | Low |### The Big Cost Traps1. **Cluster left running overnight** — a single `i3.xlarge` cluster costs ~$1/hr. Left running 12 hours = $12 wasted2. **Vector Search endpoint never deleted** — $0.38/hr × 24hr × 14 days = **$128** (32% of your budget!)3. **Multi-node cluster when single-node suffices** — 4x the cost for no benefit in these labs---

---## Step 3: Create a Cost-Optimized ClusterThis is the cluster you'll use for all labs. We configure it once, optimized forcost, and reuse it throughout the course.### UI Walkthrough1. **UI →** Left nav → **Compute**2. Click **Create compute**3. Configure exactly as follows:| Setting | Value | Why ||---------|-------|-----|| **Name** | `genai-labs` | Easy to identify || **Policy** | Unrestricted (or Personal Compute) | Full flexibility || **Cluster mode** | **Single Node** | No workers needed for labs — saves 50%+ cost || **Databricks Runtime** | **15.4 LTS ML** or latest LTS ML | ML runtime includes MLflow, langchain pre-installed || **Node type** | `m5.large` (2 vCPU, 8GB) or `i3.xlarge` | Smallest reasonable size || **Auto termination** | **20 minutes** | Shuts down after 20 min idle — critical cost saver || **Spot instances** | Enable if available | 60–90% cheaper than on-demand |4. Click **Create compute**5. Wait 3–5 minutes for startup> **Cost math:** `m5.large` ≈ $0.40/hr in DBUs. With 20-min auto-terminate,> a typical 2-hour lab session costs ~$0.80. All 35 labs ≈ $28–40.---

In [ ]:
# --------------------------------------------------------------------------
# Step 3 — Verify your compute is running
# --------------------------------------------------------------------------
# Run this cell after attaching to your cluster or serverless compute.
# If it prints without error, you're connected!

print(f"Spark version: {spark.version}")
print(f"Cluster: running and connected!")
print()

# Check compute config (handles both classic clusters and serverless)
try:
    cluster_name = spark.conf.get("spark.databricks.clusterUsageTags.clusterName")
    node_type = spark.conf.get("spark.databricks.clusterUsageTags.clusterNodeType", "unknown")
    print(f"Compute type : Classic cluster")
    print(f"Cluster name : {cluster_name}")
    print(f"Node type    : {node_type}")
except Exception:
    print(f"Compute type : Serverless")
    print(f"Cluster name : (managed by Databricks)")
    print(f"Node type    : (managed by Databricks)")

print()
print("You're ready to go!")

**Expected output (classic cluster):**```Spark version: 3.5.0Cluster: running and connected!Compute type : Classic clusterCluster name : genai-labsNode type    : m5.largeYou're ready to go!```**Expected output (serverless):**```Spark version: 3.5.0Cluster: running and connected!Compute type : ServerlessCluster name : (managed by Databricks)Node type    : (managed by Databricks)You're ready to go!```Both are fine! The key is that Spark version prints without error.---

---## Step 4: Verify Auto-Termination is SetThis is the **single most important cost guardrail**. If auto-termination is not set,your cluster runs forever and drains your credits.### How to Check1. **UI →** Left nav → **Compute**2. Click on your `genai-labs` cluster3. Look for **"Auto termination"** in the configuration panel4. It should say **"20 minutes"** (or whatever you set in Step 3)5. If it says "Never" — **edit the cluster immediately** and set it to 20 minutes### Programmatic Check

In [ ]:
# --------------------------------------------------------------------------
# Step 4 — Check auto-termination setting
# --------------------------------------------------------------------------
try:
    auto_term = spark.conf.get("spark.databricks.cluster.profile", "unknown")
    print(f"Cluster profile: {auto_term}")
except Exception:
    pass

# The most reliable way to check is via the UI.
# But we can remind ourselves:
print()
print("COST GUARDRAIL CHECKLIST:")
print("=" * 50)
print("[  ] Auto-terminate set to 20 minutes")
print("[  ] Single-node mode (no workers)")
print("[  ] Spot instances enabled (if available)")
print("[  ] Using smallest viable node type")
print()
print("Go to Compute -> click your cluster -> verify these settings!")
print("This check takes 30 seconds and can save you $100+ in credits.")

**Expected output:**```COST GUARDRAIL CHECKLIST:==================================================[  ] Auto-terminate set to 20 minutes[  ] Single-node mode (no workers)[  ] Spot instances enabled (if available)[  ] Using smallest viable node typeGo to Compute -> click your cluster -> verify these settings!This check takes 30 seconds and can save you $100+ in credits.```---

---## Step 5: Set Up Budget TrackingYou need to know how fast you're burning through your $400. Databricks providesa usage dashboard.### How to Check Your Usage**UI →** Click your profile icon (top right) → **Manage Account** → **Usage**This shows:- **Total DBUs consumed** — your running meter- **DBU breakdown by resource type** — where your money is going- **Daily usage trend** — are you accelerating or steady?### Budget Allocation PlanHere's how to split your $400 across 14 days:| Category | Budget | Daily Limit | Notes ||----------|--------|-------------|-------|| All-Purpose Clusters | $60 | ~$4/day | Terminate after each session || Serverless Compute | $30 | ~$2/day | Auto-managed, low risk || Vector Search | $40 | Create only when needed | **Delete endpoint after Module 04 labs** || Model Serving | $20 | Pay-per-token | Foundation Models are cheap || SQL Warehouses | $15 | Auto-stops | Low risk || **Buffer** | **$235** | — | Safety margin for mistakes |> **Key insight:** With discipline, you'll only use ~$165 of your $400.> The buffer protects you from accidents (like forgetting to delete Vector Search).---

In [ ]:
# --------------------------------------------------------------------------
# Step 5 — Budget calculator
# --------------------------------------------------------------------------
# Use this to estimate your remaining budget at any point during the course

TOTAL_CREDITS = 400.00  # USD
TRIAL_DAYS = 14

# Estimated costs per lab session (2 hours average)
cost_per_session = {
    "cluster_2hr": 0.80,         # m5.large single-node, 2 hours
    "serverless_overhead": 0.20,  # minor serverless usage
    "foundation_model_tokens": 0.10,  # LLM API calls
}
session_cost = sum(cost_per_session.values())

# Fixed costs (things that run continuously)
vector_search_per_day = 0.38 * 24  # $9.12/day when active
vector_search_days_needed = 4       # only for Module 04 labs

total_lab_sessions = 35
total_session_cost = total_lab_sessions * session_cost
total_vector_search = vector_search_per_day * vector_search_days_needed

print("BUDGET ESTIMATE")
print("=" * 50)
print(f"Total credits available:     ${TOTAL_CREDITS:.2f}")
print(f"")
print(f"Lab sessions (35 × ${session_cost:.2f}):   ${total_session_cost:.2f}")
print(f"Vector Search ({vector_search_days_needed} days):        ${total_vector_search:.2f}")
print(f"")
total_estimated = total_session_cost + total_vector_search
remaining = TOTAL_CREDITS - total_estimated
print(f"Estimated total spend:       ${total_estimated:.2f}")
print(f"Remaining buffer:            ${remaining:.2f}")
print(f"")
print(f"Daily budget (safe pace):    ${TOTAL_CREDITS / TRIAL_DAYS:.2f}/day")
print(f"")
if remaining > 100:
    print("STATUS: You have plenty of budget headroom.")
else:
    print("STATUS: Budget is tight — be extra careful with cluster uptime.")

**Expected output:**```BUDGET ESTIMATE==================================================Total credits available:     $400.00Lab sessions (35 × $1.10):   $38.50Vector Search (4 days):        $36.48Estimated total spend:       $74.98Remaining buffer:            $325.02Daily budget (safe pace):    $28.57/daySTATUS: You have plenty of budget headroom.```---

---## Step 6: Daily Cost Habits (The Rules)Follow these rules every day and you'll never run out of credits.### Before Each Lab Session1. **Start your cluster** → Compute → `genai-labs` → Start2. **Verify auto-terminate is 20 min** (check every time — settings can reset)3. Run your lab### After Each Lab Session1. **Manually terminate your cluster** → Compute → `genai-labs` → Terminate   - Don't rely on auto-terminate alone — be proactive2. **Delete any Vector Search endpoints** you created (unless you need them tomorrow)   - **UI →** Catalog → Vector Search → select endpoint → Delete3. **Check your usage** → Profile → Manage Account → Usage### The Three Golden Rules| Rule | Why | Savings ||------|-----|---------|| **Always terminate your cluster when done** | Clusters cost ~$1/hr idle | Up to $200 over 14 days || **Delete Vector Search endpoints after use** | $9/day when running 24/7 | Up to $128 over 14 days || **Use serverless for quick tests** | Starts in seconds, auto-stops | Saves cluster startup time + cost |---

In [ ]:
# --------------------------------------------------------------------------
# Step 6 — Create a reminder function you can paste into any notebook
# --------------------------------------------------------------------------

def cost_reminder():
    """Print a cost reminder. Paste this into the last cell of every lab."""
    print()
    print("=" * 55)
    print("  COST REMINDER — DO THIS BEFORE YOU CLOSE THE TAB!")
    print("=" * 55)
    print("  1. Terminate your cluster: Compute -> genai-labs -> Stop")
    print("  2. Delete Vector Search endpoints (if created)")
    print("  3. Check usage: Profile -> Manage Account -> Usage")
    print("=" * 55)

cost_reminder()

**Expected output:**```=======================================================  COST REMINDER — DO THIS BEFORE YOU CLOSE THE TAB!=======================================================  1. Terminate your cluster: Compute -> genai-labs -> Stop  2. Delete Vector Search endpoints (if created)  3. Check usage: Profile -> Manage Account -> Usage=======================================================```> **Pro tip:** Copy the `cost_reminder()` function and paste it as the last cell> in every lab notebook. It's a 5-second habit that saves real money.---

---## Step 7: Verify Your Workspace is ReadyLet's run a quick check to make sure everything works for the remaining 34 labs.

In [ ]:
# --------------------------------------------------------------------------
# Step 7 — Workspace readiness check
# --------------------------------------------------------------------------
import os

checks = {}

# Check 1: Spark is running
try:
    v = spark.version
    checks["Spark"] = f"v{v}"
except:
    checks["Spark"] = "FAILED"

# Check 2: Unity Catalog accessible
try:
    spark.sql("USE CATALOG workspace")
    checks["Unity Catalog"] = "workspace catalog accessible"
except Exception as e:
    checks["Unity Catalog"] = f"FAILED: {e}"

# Check 3: Can create schema
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.genai_labs")
    checks["Schema"] = "workspace.genai_labs ready"
except Exception as e:
    checks["Schema"] = f"FAILED: {e}"

# Check 4: Can write Delta table
try:
    test_df = spark.createDataFrame([("test", 1)], ["col1", "col2"])
    test_df.write.format("delta").mode("overwrite").saveAsTable("workspace.genai_labs._setup_test")
    spark.sql("DROP TABLE IF EXISTS workspace.genai_labs._setup_test")
    checks["Delta Write"] = "write + read + delete OK"
except Exception as e:
    checks["Delta Write"] = f"FAILED: {e}"

# Check 5: Python environment
try:
    import json, re, os
    checks["Python Stdlib"] = "json, re, os OK"
except:
    checks["Python Stdlib"] = "FAILED"

# Check 6: Databricks token available
token = os.environ.get("DATABRICKS_TOKEN") or dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)
if token:
    checks["Auth Token"] = "available (for API calls)"
else:
    checks["Auth Token"] = "NOT FOUND — LLM API calls may fail"

# Print results
print("WORKSPACE READINESS CHECK")
print("=" * 55)
all_ok = True
for check, status in checks.items():
    marker = "+" if "FAILED" not in status else "X"
    if "FAILED" in status:
        all_ok = False
    print(f"  [{marker}] {check}: {status}")

print("=" * 55)
if all_ok:
    print("  All checks passed! You're ready for Lab 00-01.")
else:
    print("  Some checks failed. Fix the issues above before continuing.")

**Expected output:**```WORKSPACE READINESS CHECK=======================================================  [+] Spark: v3.5.0  [+] Unity Catalog: workspace catalog accessible  [+] Schema: workspace.genai_labs ready  [+] Delta Write: write + read + delete OK  [+] Python Stdlib: json, re, os OK  [+] Auth Token: available (for API calls)=======================================================  All checks passed! You're ready for Lab 00-01.```If any check fails, fix it before moving on. Common fixes:- **Unity Catalog failed:** Your workspace may not have UC enabled — contact Databricks support- **Auth Token not found:** Restart your cluster; tokens are injected at cluster start- **Delta Write failed:** Check you have write permissions on `workspace.genai_labs`---

---## After Your 14-Day TrialWhen your trial ends, you are **automatically enrolled in pay-as-you-go**. This isNOT a fixed monthly fee — you only pay for what you use.### What Pay-As-You-Go Means| Question | Answer ||----------|--------|| Is there a monthly fee? | **No** — you only pay for compute you use || What's the rate? | $1.00 per DBU (same as trial) || What if I don't use it? | You pay **$0** — no usage = no charge || Will I be charged automatically? | Yes — charges go to your AWS bill || Can I cancel anytime? | Yes — no commitment, no cancellation fee |### The Real RiskIf you forget to clean up after the trial, these resources **keep running and charging**:| Forgotten Resource | Cost per Day | Cost per Month ||-------------------|-------------|----------------|| Running cluster (m5.large) | ~$10/day | ~$300/month || Vector Search endpoint | ~$9/day | ~$274/month || SQL Warehouse (idle but not stopped) | ~$5/day | ~$150/month |> **Bottom line:** If everything is stopped/deleted, pay-as-you-go costs $0.> The danger is ONLY from running compute you forgot about.### How to Cancel (Do This on Day 12–13)1. **Terminate all clusters** → **UI →** Compute → Stop/Terminate everything2. **Delete all Vector Search endpoints** → **UI →** Catalog → Vector Search → Delete3. **Stop SQL Warehouses** → **UI →** SQL Warehouses → Stop all4. **Cancel the subscription:**   - Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)   - Find **Databricks Data Intelligence Platform**   - Click **Manage** → **Cancel subscription**   - Your workspace will be deactivated (data retained for 30 days)### If You Want to Keep Learning- **Option A:** Stay on pay-as-you-go — just be disciplined about terminating resources- **Option B:** Cancel AWS Marketplace and sign up for [Databricks Free Edition](https://www.databricks.com/try-databricks) — limited but free forever (serverless only, quota-limited)- **Option C:** Cancel everything and rely on your notes + solutions for exam review### Set Your Calendar Reminder NOWAdd these reminders:- **Day 7:** Check usage in Account Console → are you on track?- **Day 12:** Final lab push — finish any remaining labs- **Day 13:** Clean up ALL resources (clusters, Vector Search, warehouses)- **Day 14:** Cancel subscription if you're done---

## Exam Tips & Traps| # | Tip | Why It Matters ||---|-----|---------------|| 1 | **Know the three compute types:** Clusters, Serverless, SQL Warehouses | Exam tests when to use each || 2 | **Clusters need manual termination or auto-terminate** | If you know this for cost, you know it for the exam || 3 | **Vector Search endpoints are separate compute** | Not part of your cluster — they run independently || 4 | **Foundation Model APIs are pay-per-token** | No endpoint to manage — just call the API || 5 | **Unity Catalog is required** for model registry, governance, and lineage | Every deployment lab depends on UC |---

## Key Takeaways### Setup- **AWS Marketplace trial** gives you the full Databricks platform for 14 days with $400 in credits- That's enough for all 35 labs with ~$235 buffer if you follow cost discipline### Cost Guardrails- **Auto-terminate clusters at 20 minutes** — the single most important setting- **Delete Vector Search endpoints** when not actively using them- **Use single-node clusters** — multi-node is overkill for these labs- **Check usage daily** — Profile → Manage Account → Usage### The Three Golden Rules1. Terminate your cluster when done2. Delete Vector Search endpoints after use3. Use serverless for quick tests---**Next Lab →** [Lab 00-01: Workspace Orientation](./01_workspace_orientation.ipynb)— Now that your workspace is running, let's learn how to navigate it.